In [1]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [2]:
class Model(nn.Module):
    def __init__(self, n_tabular_features):
        super().__init__()

        # Image structure
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.dropout1 = nn.Dropout(0.2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.dropout2 = nn.Dropout(0.3)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.fc_image1 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(0.5)
        self.fc_image2 = nn.Linear(64, 32)
        self.fc_image3 = nn.Linear(32, 16)

        # Tabular structure
        self.fc_tabular1 = nn.Linear(n_tabular_features, 32)
        self.fc_tabular2 = nn.Linear(32, 16)
        self.fc_tabular3 = nn.Linear(16, 8)
        self.fc_tabular4 = nn.Linear(8, 4)

        # Concatenated structure
        self.fc1 = nn.Linear(20, 8) # why 20 ? ---> 16 for Image and 4 for Tabular
        self.fc2 = nn.Linear(8, 1)

    def forward(self, Image, Tabular):
        # Image
        x = self.conv1(Image)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = nn.AdaptiveAvgPool2d((1, 1))(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc_image1(x)
        x = F.relu(x)
        x = self.dropout3(x)
        x = self.fc_image2(x)
        x = F.relu(x)
        x = self.fc_image3(x)
        x = F.relu(x)

        #Tabular
        t = self.fc_tabular1(Tabular)
        t = F.relu(t)
        t = self.fc_tabular2(t)
        t = F.relu(t)
        t = self.fc_tabular3(t)
        t = F.relu(t)
        t = self.fc_tabular4(t)
        t = F.relu(t)

        # Combine
        combined = torch.cat([x, t], dim=1)

        # Price
        x = self.fc1(combined)
        x = F.relu(x)
        x = self.fc2(x)

        return x

In [3]:
class HouseDataset(Dataset):
    def __init__(self, dataframe, image_dir, tabular_cols, transform=None):
        self.original_indices = dataframe.index.to_numpy()
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.tabular_cols = tabular_cols
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        tabular = torch.tensor(
            row[self.tabular_cols].values.astype(np.float32),
            dtype=torch.float32
        )

        house_id = int(self.original_indices[idx]) + 1

        image_names = [
            f"{house_id}_bathroom.jpg",
            f"{house_id}_bedroom.jpg",
            f"{house_id}_frontal.jpg",
            f"{house_id}_kitchen.jpg"
        ]

        input_images = []
        for image_name in image_names:
            image_path = os.path.join(self.image_dir, image_name)
            image = cv2.imread(image_path)
            image = cv2.resize(image, (64, 64))
            input_images.append(image)

        output_image = np.zeros((128, 128, 3), dtype=np.uint8)

        output_image[0:64, 0:64] = input_images[0]    # bathroom
        output_image[0:64, 64:128] = input_images[1]  # bedroom
        output_image[64:128, 64:128] = input_images[2] # frontal
        output_image[64:128, 0:64] = input_images[3]   # kitchen

        output_image = cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB)
        output_image = torch.tensor(output_image, dtype=torch.float32) / 255.0
        output_image = output_image.permute(2, 0, 1)

        price = torch.tensor(row['price'], dtype=torch.float32)

        return output_image, tabular, price

In [4]:
def train(model, device, train_loader, optimizer, criterion, epoch):
    model.train()
    running_loss = 0.0
    for batch_idx, (images, tabular, target) in enumerate(train_loader):
        images = images.to(device)
        tabular = tabular.to(device)
        target = target.to(device)
        optimizer.zero_grad()
        output = model(images, tabular)
        output = output.squeeze(1)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if batch_idx % 10 == 0:
            print(
                f"Train Epoch: {epoch} "
                f"[{batch_idx * len(images)}/"
                f"{len(train_loader.dataset)} "
                f"({100. * batch_idx / len(train_loader):.0f}%)] "
                f"Loss: {loss.item():.6f}"
            )

    return running_loss / len(train_loader)

In [5]:
def test(model, device, test_loader, criterion):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    total_mape = 0.0
    total_samples = 0
    all_predictions = []
    all_targets = []
    with torch.no_grad():
        for images, tabular, target in test_loader:
            images = images.to(device)
            tabular = tabular.to(device)
            target = target.to(device)
            output = model(images, tabular).squeeze(1)
            loss = criterion(output, target)
            total_loss += (loss.item() * len(target))
            total_mae += (torch.abs(output - target).sum().item())
            total_mape += (torch.abs((output - target) / target).sum().item())
            all_predictions.extend(output.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            total_samples += len(target)

    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)
    mse = total_loss / total_samples
    rmse = mse ** 0.5
    mae = total_mae / total_samples
    mape = (total_mape / total_samples) * 100
    ss_res = np.sum((all_targets - all_predictions) ** 2)
    ss_tot = np.sum((all_targets - np.mean(all_targets)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    print(
        f"\nTest set:"
        f"\nMSE:   {mse:.6f}"
        f"\nRMSE:  {rmse:.6f}"
        f"\nMAE:   {mae:.6f}"
        f"\nMAPE:  {mape:.2f}%"
        f"\nR²:    {r2:.4f}\n"
    )

    return mse, rmse, mae, mape, r2

In [ ]:
torch.manual_seed(42)
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')

df = pd.read_csv('HousesInfo.txt', header=None, sep=' ', names=['bedroom', 'bathroom', 'area', 'zipcode', 'price'])

df = pd.get_dummies(df, columns=['zipcode'])

tabular_cols = [c for c in df.columns if c != 'price']
n_tabular_features = len(tabular_cols)

print(f"Number of tabular features: {n_tabular_features}")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

image_dir = 'data'

train_dataset = HouseDataset(train_df, image_dir, tabular_cols)
test_dataset = HouseDataset(test_df, image_dir, tabular_cols)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

model = Model(n_tabular_features).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

epochs = 50
for epoch in range(1, epochs + 1):
    train_loss = train(model, device, train_loader, optimizer, criterion, epoch)
    scheduler.step()
    mse, rmse, mae = test(model, device, test_loader, criterion)

torch.save(model.state_dict(), 'model/house_price_model.pth')
print("Model saved.")

Number of tabular features: 52


## Test the data on trained model

In [12]:
tmodel = Model(n_tabular_features).to(device)

model.load_state_dict(torch.load('model/house_price_model.pth', map_location=device))
model.eval()
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

mse, rmse, mae, mape, r2 = test(model, device, test_loader, criterion)

print(f"R² = {r2:.3f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")


Test set:
MSE:   130789278117.084106
RMSE:  361648.003060
MAE:   266874.670561
MAPE:  108.74%
R²:    0.0554

R² = 0.055
MSE: 130789278117.08
RMSE: 361648.00
MAE: 266874.67


In [13]:
def price_to_class(price):
    if price < 500000:
        return 0
    elif price < 750000:
        return 1
    elif price < 1000000:
        return 2
    else:
        return 3

In [14]:
def test_classification_accuracy(model, device, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, tabular, target in test_loader:
            images = images.to(device)
            tabular = tabular.to(device)
            target = target.to(device)

            output = model(images, tabular).squeeze(1)

            pred_classes = torch.tensor([price_to_class(p) for p in output.cpu().numpy()])
            true_classes = torch.tensor([price_to_class(t) for t in target.cpu().numpy()])

            correct += (pred_classes == true_classes).sum().item()
            total += len(target)

    accuracy = correct / total
    print(f"Classification Accuracy: {accuracy:.4f}")
    return accuracy

In [15]:
test_classification_accuracy(model, device, test_loader)

Classification Accuracy: 0.4766


0.4766355140186916